# 내 업무 Agent 직접 완성하기

조회 도구·Agent·업무 분기를 직접 만들고 제공 그래프·루프에 연결합니다. 전체 실행 구조의 재구현은 선택 심화입니다. 각 함수의 NotImplementedError를 자신의 구현으로 바꿉니다. 정답은 별도 노트북에 있습니다. 실행 출력은 제출 전에 키나 민감한 입력이 없는지 확인합니다.

웹 교재의 **직접 완성하기**에 재료, 입출력 표, API 힌트와 반례가 있습니다. 이 노트북은 그중 1~3단계를 진행합니다. 셀 순서를 지키고 마지막에는 커널 재시작 후 전체 실행합니다.

In [ ]:
from pathlib import Path
import os, sys, json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "build_lab" / "materials.py").is_file():
    raise RuntimeError("workshop/notebooks에서 이 노트북을 여십시오.")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END
from build_lab.materials import POLICIES, Inquiry, inspect_draft, get_model, trace_messages
print("실행 위치:", root)
print("정책 주제:", list(POLICIES))

## 1A. 조회 도구

공백을 제거한 topic으로 정책을 조회합니다. JSON 문자열에 found, topic, policy를 넣습니다. 없는 정책은 False와 None입니다. dict.get, str.strip, json.dumps가 재료입니다. 먼저 계정과 없는업무의 결과를 예상합니다.

In [ ]:
def lookup_policy(topic: str) -> str:
    """Look up the current internal policy by topic, such as 정산 or 계정."""
    # 공백을 제거한 topic으로 POLICIES를 조회합니다. JSON 문자열을 반환합니다.
    raise NotImplementedError("1A: found, topic, policy를 반환하는 조회 도구를 구현하십시오.")

In [ ]:
for topic in ["정산", " 계정 ", "없는업무"]:
    data = json.loads(lookup_policy(topic))
    print(data)
    assert data["found"] == (topic.strip() in POLICIES)
    assert data["topic"] == topic.strip()

## 1B. LangChain Agent

model, tools, system_prompt를 지정해 Agent를 반환합니다. 제공된 model과 policy_tool을 사용합니다. 도구 조회 시점과 정책이 없을 때의 행동을 지침에 적습니다. 다음 셀은 실제 모델 호출입니다.

In [ ]:
def build_agent(model, policy_tool):
    # model·tools·system_prompt를 지정합니다. 실제 호출은 호출자가 수행합니다.
    raise NotImplementedError("1B: create_agent로 도구를 가진 Agent를 구성하십시오.")

In [ ]:
model = get_model()
agent = build_agent(model, lookup_policy)
result = agent.invoke({"messages": [{"role": "user", "content": '{"topic":"계정"}'}]}, config={"recursion_limit": 12})
for entry in trace_messages(result["messages"]):
    print(json.dumps(entry, ensure_ascii=False, indent=2))

**기록:** 도구 요청 / 도구 반환 / 최종 답변에서 같은 ID와 팀을 찾아 적습니다. 답변이 틀리면 도구와 지침 중 어느 쪽을 먼저 볼지 설명합니다. 확인 후 없는업무를 추가 실행합니다.

## 2. 업무 Graph

공통 과정은 route_inquiry를 작성하고 제공 build_workflow를 읽습니다. 전체 노드 연결 구현은 선택 심화입니다. lookup 다음에 조건을 검사합니다. 정책이 있고 contact.strip()이 비어 있지 않으면 draft→review, 그렇지 않으면 ask로 끝냅니다. 노드는 변경할 State 필드만 반환합니다. visited에 실제 노드 이름을 순서대로 남깁니다. 웹 교재의 노드별 입출력 표를 보고 작성합니다.

In [ ]:
def route_inquiry(state):
    """정책과 회신 대상을 읽고 draft 또는 ask로 분기합니다."""
    # 정책이 있어도 회신 대상이 없으면 생성하면 안 됩니다.
    raise NotImplementedError("2: 정책 유무와 공백 연락처를 검사하는 분기를 구현하십시오.")

def build_workflow(lookup, generate, refine, limit=2):
    """제공 그래프에 학생 분기를 연결합니다. guided.py의 노드·간선을 읽습니다."""
    from build_lab.guided import build_workflow as assemble
    return assemble(lookup, generate, refine, limit, router=route_inquiry)

In [ ]:
def generate(topic):
    reply = agent.invoke({"messages": [{"role": "user", "content": json.dumps({"topic": topic}, ensure_ascii=False)}]}, config={"recursion_limit": 12})
    return reply["messages"][-1].content

# 수정 루프를 배우기 전에는 실제 초안을 한 번 검토합니다.
def inspect_once(draft, data, limit):
    feedback = inspect_draft(draft, data)
    return {"status": "held" if feedback else "passed", "draft": draft,
            "history": [{"attempt": 0, "draft": draft, "feedback": feedback}]}

graph = build_workflow(lookup_policy, generate, inspect_once)
missing = graph.invoke({"topic": "정산", "contact": "   "})
print(missing)
assert missing["decision"] == "ask"
assert missing["visited"] == ["lookup", "ask"]
normal = graph.invoke({"topic": "계정", "contact": "user@example.test"})
print(normal)

**기록:** 공백 문의에서 모델을 호출했는지 확인합니다. 조건을 잘못 연결한 경우와 정책 조회가 실패한 경우를 구분합니다. 연락처 확인을 조회 앞으로 옮긴다면 어떤 노드가 생략되는지 그립니다.

## 3. 제공 수정 Loop 관찰

공통 과정에서는 아래 연결 함수를 그대로 사용하고 guided.py를 읽습니다. 직접 재구현하려는 경우에만 다음 계약을 구현합니다. 성공→정체→예산 순서로 검사합니다. status는 passed, stalled, held입니다. history에는 attempt, draft, feedback을 남깁니다. 최초 검토는 attempt=0입니다. limit은 0~5 정수이며 bool은 거부합니다. 수정에는 이번 초안의 피드백을 전달합니다.

In [ ]:
def refine_answer(draft, data, revise, limit=2):
    """제공 루프를 사용합니다. 전체 알고리즘 작성은 선택 심화입니다."""
    from build_lab.guided import refine_answer as refine
    return refine(draft, data, revise, limit)

In [ ]:
feedback_inputs = []
data = json.loads(lookup_policy("계정"))
def revise(draft, feedback):
    feedback_inputs.append({"draft": draft, "feedback": feedback})
    return model.invoke("규정에 따라 초안을 수정하십시오. 담당 팀과 근거 ID를 포함하십시오.\n" +
                        json.dumps({"policy": data, "draft": draft, "feedback": feedback}, ensure_ascii=False)).content

repaired = refine_answer("확인 완료", data, revise, 2)
print(json.dumps(repaired, ensure_ascii=False, indent=2))
print("실제로 전달한 수정 입력:", feedback_inputs)
assert feedback_inputs, "이 초안은 기준이 부족하므로 수정 호출이 필요합니다."
assert repaired["history"][0]["draft"] == "확인 완료"

## 연결하고 설명합니다

앞에서 만든 그래프에 제공 수정 루프를 연결합니다. 결과가 통과하지 않으면 기준을 낮추지 말고 history의 실패와 수정 입력을 읽습니다.

In [ ]:
def refine(draft, data, limit):
    return refine_answer(draft, data, revise, limit)
app = build_workflow(lookup_policy, generate, refine, limit=2)
final = app.invoke({"topic": "계정", "contact": "user@example.test"})
print(json.dumps(final, ensure_ascii=False, indent=2))

## 직접 만든 반례와 다음 단계

1. 같은 실패 초안이 돌아왔을 때 왜 멈추는지 설명합니다.
2. 근거 ID가 있지만 사실과 다른 문장 하나를 만듭니다. 현재 검토 기준이 이를 잡는지 확인합니다.
3. 다른 주제를 실행한다면 revise가 참조하는 policy도 해당 주제로 바꿉니다. 계정 data를 그대로 사용하면 다른 요청의 근거로 수정하게 됩니다.
4. 직접 작성한 lookup_policy, build_agent, route_inquiry만 build_lab/student.py의 같은 이름 함수로 옮깁니다. 제공 build_workflow, refine_answer는 그대로 둡니다. 함수 정의를 복사하고 import는 제공 파일을 유지합니다.
5. 웹 교재의 MCP·A2A 단계에서 나머지 두 함수를 구현하고 전체 프로젝트를 실행합니다.

제출 기록: 첫 실패 / 수정 이유 / 실행 경로 / 검토의 한계 / 새 반례. 셀 실행 성공만으로 문장의 정확성을 판정하지 않습니다.

## 공통 활동: 코딩 Harness 활용

[웹 활동](https://yo-sure.github.io/deepagents-handson/workshop/engineering#task)과 `../build_lab/HARNESS_WORKSHEET.md`를 읽고 F1/D1 검수 사례의 시작·작업 선택·중단·역할 의존성·산출물 계약을 작성합니다. 계정이 있으면 실제 하네스로 확인하고, 없으면 웹의 수업용 검수 사례와 풀이를 검토합니다. 이미 공백 분기가 맞으면 수정 없이 증거를 남깁니다. 활동지를 제출 기록에 포함합니다. 앞 단계에서 막히면 `../README.md`의 복귀 절차에 따라 자신의 작업을 보존한 뒤 미완료 함수만 기준 구현으로 보완하고 제공받은 부분을 기록합니다.